# STRF and LN model performance on Wehr's recordings (Asari & Zador 2009, Figure 4)

**Paper.** Asari H, Zador AM (2009). Long-lasting context dependence constrains neural encoding
models in rodent auditory cortex. *J Neurophysiol* 102:2638-2656.
[doi:10.1152/jn.00577.2009](https://doi.org/10.1152/jn.00577.2009)

**Dataset.** CRCNS ac-1, [doi:10.6080/K0KW5CXR](https://doi.org/10.6080/K0KW5CXR), the `wehr/`
folder: Michael Wehr's whole-cell recordings of A1 neurons to natural sounds (Machens et al. 2004),
one file per trial, with the stimuli and the authors' `STRFanalysis.m`. Streamed from the public
S3 bucket as in `reproduce_asari_2009.ipynb`, which gives the version (unversioned; MD5 sums of
the zip files in `asari.MD5`).

**Claims.** `claims/asari-2009.yaml`. This notebook tests one claim, fig4-strf-performance-low; the
others are in `reproduce_asari_2009.ipynb`.

| claim | tested here |
|---|---|
| fig4-strf-performance-low | tested |

Methods (Eqs. 46-53 and the claim's criterion) and where each detail comes from are in the
docstring of `wehr_strf.py`. The paper reanalysed 20 of these cells without naming them. Two
readings of the cells: the 16 cells with natural sounds heard at least twice, using only those
stimuli (`repeated`, main), and all 25 cells with every stimulus, a stimulus heard once given half
its total power as predictable power, as `STRFanalysis.m` does (`code`).

In [ ]:
import os
import time
import resource
from concurrent.futures import ProcessPoolExecutor

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt

import asari
import wehr_strf as ws

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 30)
T0 = time.time()
A = asari.Archive()
cells = ws.wehr_cells(A)
print("cells in wehr/Summary.mat:", len(cells))
print("windows (ms):", ws.WINDOWS_MS, "| parameters per model:", len(ws.FREQS) * 45)
print("centre frequencies (kHz):", np.round(ws.FREQS / 1000, 2))
with ProcessPoolExecutor(3) as ex:
    done = list(ex.map(ws.run_cell, cells))
info = pd.DataFrame([d["info"] for d in done])
res = pd.DataFrame([r for d in done for r in d["rows"]])
print(f"analysed in {(time.time() - T0) / 60:.1f} min; peak memory of this process "
      f"{resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")

## Data check

In [ ]:
print("experiments:", dict(pd.Series([c["experiment_description"] for c in cells]).value_counts()))
info["stimuli_heard_once"] = info.n_stimuli - info.n_repeated
info["trials_range"] = info.trials.map(lambda t: (min(t), max(t)))
info["stimulus_s"] = info.seconds.map(lambda s: (round(min(s), 2), round(max(s), 2)))
print(info[["cell", "n_stimuli", "n_repeated", "stimuli_heard_once", "trials_range", "stimulus_s", "recording_types",
            "sf", "stim_sf", "reliability", "resp_range", "spec_range"]].round(3).to_string())
print("\nrecording types:", sorted({t for x in info.recording_types for t in x}),
      "| response sampling rates (Hz):", sorted({s for x in info.sf for s in x}),
      "| stimulus sampling rates (Hz):", sorted({s for x in info.stim_sf for s in x}))
print("cells with stimuli heard at least twice:", int((info.n_repeated > 0).sum()), "of", len(info))
print("reliability is the predictable over the total power (Eq. 52) of the repeated stimuli, 4-ms bins;"
      " resp_range in mV after centring each trial; spec_range in dB after centring each frequency")

Every file is a whole-cell recording at 4 kHz, one trial per file, with one trigger for the
stimulus. The stimuli are sampled at 97.656 kHz. The predictable fraction of the response power
(reliability above) is well above zero in every cell with repeated stimuli, which is consistent
with the traces and the stimuli sharing a clock: a misaligned trial would lower it.

## fig4-strf-performance-low

> Linear STRF models of 20 A1 cells from Machens et al. (2004) predicted less than about 20% of the
> predictable response power, and extending the window from 180 ms to about 4 s did not improve
> them significantly, with or without a static nonlinearity; the LN model did only slightly better.

Criterion: reproduced when, under one reading, the mean performance is at most 0.22 at every
window for STRF and LN, and the one-sided paired Wilcoxon test (validation performance at 4052 ms
against 180 ms, over cells, at 0.05) finds no improvement for either. Readings: validation (lower)
or training (upper) performance; spectrogram window 10 or 20 ms; contiguous folds or folds by
stimulus; the cells (`repeated`, `code`); and two details of this implementation, the samples
scored and validation pooled over folds or averaged over folds.

In [ ]:
def readings(res):
    out = {}
    for cfg, g in res[res.reading == "repeated"].groupby("config"):
        out[(cfg, "repeated")] = g
    code_rows = res[(res.config == "main") & (res.reading == "code")]
    same = res[(res.config == "main") & (res.reading == "repeated") & ~res.cell.isin(code_rows.cell)]
    out[("main", "code")] = pd.concat([same, code_rows])
    return out

def summarize(g, val):
    rows = []
    for model in ["strf", "ln"]:
        col = f"{model}_{val}"
        p = g.pivot(index="cell", columns="window_ms", values=col)
        w = stats.wilcoxon(p[4052], p[180], alternative="greater")
        fr = stats.friedmanchisquare(*[p[c] for c in p.columns])
        rows.append(dict(model=model.upper(), measure=val, n_cells=len(p), **{f"{c}": p[c].mean() for c in p.columns},
                         max_mean=p.mean().max(), wilcoxon_p=w.pvalue, median_gain_4052_vs_180=(p[4052] - p[180]).median(),
                         friedman_p=fr.pvalue))
    return rows

R = readings(res)
rows = []
for (cfg, cells_reading), g in R.items():
    for val in ["valid", "train", "valid_foldmean"]:
        for r in summarize(g, val):
            rows.append(dict(config=cfg, cells=cells_reading, **r))
tab = pd.DataFrame(rows)
print("mean performance over cells at each window (ms), the largest mean, and the tests of 4052 against 180 ms")
print(tab.round(3).to_string())

In [ ]:
crit = []
for (cfg, cr, val), g in tab.groupby(["config", "cells", "measure"]):
    d = g.set_index("model")
    crit.append(dict(config=cfg, cells=cr, measure=val, max_mean_STRF=d.max_mean["STRF"], max_mean_LN=d.max_mean["LN"],
                     wilcoxon_p_STRF=d.wilcoxon_p["STRF"], wilcoxon_p_LN=d.wilcoxon_p["LN"],
                     low=(d.max_mean <= 0.22).all(), no_improvement=(d.wilcoxon_p >= 0.05).all()))
crit = pd.DataFrame(crit)
crit["criterion_met"] = crit.low & crit.no_improvement
print(crit.round(3).to_string())
main = R[("main", "repeated")]
gain = (main.ln_valid - main.strf_valid)
print("\nLN minus STRF validation performance (main reading), over cells and windows: mean", round(gain.mean(), 3),
      "| median", round(gain.median(), 3), "| range", (round(gain.min(), 3), round(gain.max(), 3)))
pc = main.pivot(index="cell", columns="window_ms", values="strf_valid")
print("STRF validation performance per cell, range over cells at each window:")
print(pd.DataFrame({"min": pc.min(), "max": pc.max()}).round(3).T.to_string())
print("chosen ridge value at a grid edge:", int(res.ridge_at_edge.sum()), "of", len(res), "cell x window x reading fits")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
for model, c in [("strf", "0.6"), ("ln", "k")]:
    p = main.pivot(index="cell", columns="window_ms", values=f"{model}_valid").mean()
    q = main.pivot(index="cell", columns="window_ms", values=f"{model}_train").mean()
    ax.plot(np.array(p.index) / 1000, p.values, "x-", color=c, label=f"{model.upper()} validation")
    ax.plot(np.array(q.index) / 1000, q.values, "o--", mfc="none", color=c, label=f"{model.upper()} training")
ax.set_xlabel("window length (s)"); ax.set_ylabel("model performance (Eq. 53)"); ax.set_ylim(-0.1, 1)
ax.legend(fontsize=8); plt.show()
print("plotted means range from", round(min(main[[c for c in main.columns if c.endswith(('_valid', '_train'))]].groupby(main.window_ms).mean().min()), 3),
      "to", round(max(main[[c for c in main.columns if c.endswith(('_valid', '_train'))]].groupby(main.window_ms).mean().max()), 3), "(all inside the axes)")

**Result: not reproduced (resting on the pooled validation performance, the cells and the test).** The performance is low, as stated: with validation pooled over the folds, the mean over the 16 cells is at most 0.198 for STRF and 0.171 for LN at any window (main reading), and at most 0.208 and 0.191 in every other reading, all under 0.22; the training (upper) estimates reach 0.29 to 0.50, so the paper's 20% fits the validation estimate. But the STRF does improve with the window: its validation performance rises from 0.163 at 180 ms to 0.196 at 4052 ms, a median gain of 0.024 per cell, and the one-sided Wilcoxon test finds the improvement in every pooled reading (p = 0.000 to 0.042). The LN model shows no significant improvement (p = 0.106 to 0.450), and it does worse than the STRF, not slightly better (mean difference -0.043). Over all 25 cells, with a stimulus heard once given half its total power as predictable power, the results are the same (STRF at most 0.183, Wilcoxon p = 0.000). The criterion is met in one combination only, folds by stimulus with validation averaged over folds as `STRFanalysis.m` averages it (STRF at most 0.129, p = 0.202; LN p = 0.450); with the code's contiguous folds the fold average is unstable (STRF means up to 0.83), as folds of a stimulus heard twice can hold little predictable power. The chosen ridge value is inside the grid in all 675 fits.

## Summary

| claim | outcome |
|---|---|
| fig4-strf-performance-low | not reproduced (assumptions: pooled validation performance, the cells, the test) |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the differences from the paper's method.

Differences from the paper's method: the paper reanalysed 20 of Wehr's cells without naming them; the data set holds 25, of which 16 have natural sounds heard at least twice, and both sets are run. The spectrogram, binning, ridge grid, lowess and folds are reimplemented from the Methods and `STRFanalysis.m` (see `wehr_strf.py`); the validation performance is pooled over the validation folds, with the code's per-fold average shown. The test of "did not improve significantly" is the claims file's.

What was found about the data set: Wehr's 25 cells are all whole-cell recordings at 4 kHz with one trial per file; 9 cells have only stimuli heard once, which give no estimate of the predictable power.

In [ ]:
print(f"total run time {(time.time() - T0) / 60:.1f} min; peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.1f} GB")